# Practica 2.1
# Equipo los Morgue
# 7BM1

- García Martínez Carla Edith 
- Mena Fernández Sofia 
- Suárez Ramírez Gustavo 
- Varillas Figueroa Enrique Uriel 
- García Carrizoza André

# 1. Crear las carpetas del proyecto


In [ ]:
# Importamos la clase Path del módulo nativo pathlib. 
# Esto nos permite manejar y manipular rutas de archivos y directorios 
# de forma orientada a objetos (independientemente de si es Windows, Mac o Linux).
from pathlib import Path

# Definimos la ruta absoluta de la carpeta principal del proyecto.
# Path.cwd() obtiene el directorio de trabajo actual (Current Working Directory).
# El operador '/' está sobrecargado en pathlib para concatenar carpetas fácilmente.
carpeta_proyecto = Path.cwd() / "Reconocimiento_Locutor"  

# Definimos las rutas para las subcarpetas específicas del proyecto.
# A partir de la carpeta principal, creamos una para los datos y otra para los resultados.
carpeta_datos = carpeta_proyecto / "datos"  
carpeta_resultados = carpeta_proyecto / "resultados"  

# Ejecutamos la creación física de los directorios en el disco duro.
# El método .mkdir() crea la carpeta.
# parents=True: Asegura que si una carpeta padre en la ruta no existe, también la cree.
# exist_ok=True: Evita que el código lance un error (FileExistsError) si la carpeta ya había sido creada antes.
carpeta_proyecto.mkdir(parents=True, exist_ok=True)  
carpeta_datos.mkdir(parents=True, exist_ok=True)  
carpeta_resultados.mkdir(parents=True, exist_ok=True)  

# Imprimimos en la salida estándar (consola) las rutas finales.
# Esto sirve como validación visual para asegurar que las variables apuntan a donde esperamos.
print("Carpeta del proyecto:", carpeta_proyecto)
print("Carpeta de datos:", carpeta_datos)
print("Carpeta de resultados:", carpeta_resultados)

# 2. Descargar el subconjunto de LibriSpeech dev-clean

In [ ]:
# Importamos el módulo urllib.request, que pertenece a la biblioteca estándar de Python.
# Este módulo contiene funciones y clases que nos permiten abrir y leer URLs,
# lo cual es ideal para realizar descargas de archivos desde internet.
import urllib.request

# Definimos la URL directa desde donde vamos a descargar nuestro conjunto de datos de audio.
url_descarga = ("https://openslr.elda.org/resources/12/dev-clean.tar.gz")

# Definimos las rutas de destino utilizando la variable 'carpeta_datos' (creada en la celda anterior).
# archivo_comprimido: Será el nombre y ruta final de nuestro archivo una vez descargado completamente.
archivo_comprimido = carpeta_datos / "dev-clean.tar.gz"  
# archivo_temporal: Archivo intermedio que se usará mientras los datos se están bajando.
# Esto es una buena práctica para evitar tener un archivo corrupto si la descarga se interrumpe.
archivo_temporal = carpeta_datos / "dev-clean.tar.gz.part" 

# Comprobamos si el archivo final ya existe en nuestro disco duro mediante el método .exists() de pathlib.
if archivo_comprimido.exists():
    # Si existe, evitamos volver a descargarlo (ahorrando tiempo y ancho de banda).
    print("El archivo ya existe:", archivo_comprimido) 
else:
    # Si no existe, comenzamos el proceso de descarga.
    print("Descargando LibriSpeech dev-clean...")

    # Utilizamos un bloque try-except para manejar posibles errores de conexión o escritura.
    try:
        # urllib.request.urlretrieve obtiene el contenido de la URL y lo guarda directamente en disco.
        # Aquí lo guardamos primero bajo el nombre temporal (.part).
        urllib.request.urlretrieve(url_descarga, archivo_temporal) 

        # Una vez que la descarga de urlretrieve finaliza con éxito, 
        # reemplazamos/renombramos el archivo temporal para que adquiera su nombre definitivo.
        archivo_temporal.replace(archivo_comprimido) 
        print("Descarga finalizada.")

    # Si ocurre cualquier excepción (error de red, tiempo de espera, disco lleno, etc.).
    except Exception:
        print("La descarga no terminó")
        # El comando 'raise' relanza el error original para que la ejecución se detenga 
        # y podamos ver en la consola exactamente qué falló.
        raise  

# Una vez asegurada la existencia del archivo (ya sea bajado ahora o previamente),
# usamos .stat() que devuelve información del sistema sobre el archivo.
# .st_size extrae el peso exacto del archivo en bytes.
tamano_bytes = archivo_comprimido.stat().st_size  

# Convertimos los bytes a Megabytes (MB) para que sea humanamente legible,
# dividiendo la cantidad de bytes entre 1,000,000.
tamano_nb = tamano_bytes / 1_000_000  

# Imprimimos la ruta del archivo y su tamaño calculado.
print("Archivo:", archivo_comprimido)
# Utilizamos f-strings para formatear el texto. 
# {tamano_nb:.2f} indica que queremos mostrar la variable flotante con solo 2 decimales.
print(f"Tamaño: {tamano_nb:.2f} MB")

# 3. Verificar la integridad del archivo descargado

In [ ]:
# Se importa la biblioteca hashlib para aplicar algoritmos de hash seguros (como MD5).
# Esto nos servirá para crear una "huella digital" de nuestro archivo descargado.
import hashlib 
# Se importa la biblioteca urllib.request para abrir URLs y leer datos directamente desde internet.
import urllib.request 

# Se define la URL que contiene un archivo de texto con las firmas (hashes) oficiales del dataset.
url_verificacion = "https://openslr.elda.org/resources/12/md5sum.txt"  

# Usamos el bloque 'with' para abrir la URL de manera segura, asegurando que la conexión se cierre al terminar.
# Se añade un timeout de 30 segundos para evitar que el programa se quede congelado si no hay respuesta del servidor.
with urllib.request.urlopen(url_verificacion, timeout=30) as respuesta: 
    # Leemos la respuesta HTTP cruda en bytes.
    contenido = respuesta.read()

# Se decodifica el contenido en bytes a una cadena de texto (string) utilizando codificación UTF-8, 
# que es el estándar para texto en internet.
texto_verificacion = contenido.decode("utf-8")  

# Inicializamos la variable 'md5_esperado' como 'None'. 
# Aquí guardaremos el hash oficial una vez que lo encontremos en el texto.
md5_esperado = None 

# Iteramos sobre el texto descargado línea por línea.
# El método .splitlines() separa el texto gigantesco en una lista de líneas individuales.
for linea in texto_verificacion.splitlines(): 
    # .split() divide la línea en una lista usando los espacios en blanco como separador.
    # Usualmente las líneas tienen el formato: "hash_md5 nombre_del_archivo"
    campos = linea.split()  

    # Verificamos que la línea realmente contenga dos elementos (el hash y el nombre).
    # Esto evita errores si el archivo de texto tiene líneas vacías o mal formateadas.
    if len(campos) != 2:  
        continue  # Si no tiene 2 campos, saltamos esta iteración y pasamos a la siguiente línea.

    # Extraemos el nombre del archivo (segundo campo en la lista).
    # .lstrip("*") elimina el asterisco inicial si es que el archivo de verificación lo incluye.
    nombre_archivo = campos[1].lstrip("*")  
    # Si la ruta tiene directorios (ej. carpeta/archivo.tar.gz), nos quedamos solo con el nombre del archivo.
    nombre_archivo = nombre_archivo.split("/")[-1]  

    # Comparamos el nombre que acabamos de leer con el nombre de nuestro archivo local.
    # archivo_comprimido.name extrae solo el nombre del archivo de la ruta en pathlib (dev-clean.tar.gz).
    if nombre_archivo == archivo_comprimido.name:  
        # Si coinciden, extraemos el primer campo (el hash oficial) y lo convertimos a minúsculas por seguridad.
        md5_esperado = campos[0].lower()  
        # Como ya encontramos lo que buscábamos, rompemos el ciclo 'for' para no gastar más tiempo de procesamiento.
        break  
    
# Validamos si logramos encontrar el hash. Si sigue siendo 'None', significa que nuestro archivo 
# no estaba listado en el documento oficial, por lo que lanzamos un error que detiene el código.
if md5_esperado is None:  
    raise ValueError("No se encontró la suma de comprobación de dev-clean.")

# Creamos un objeto 'verificador' con el algoritmo MD5 para calcular el hash de nuestro archivo local.
verificador = hashlib.md5()  
# Definimos un tamaño de bloque de 1 Megabyte (1024 bytes * 1024 bytes).
# Es crucial leer el archivo en trozos (bloques) para evitar cargar todo de golpe en la memoria RAM.
tamano_bloque = 1024*1024  

# Abrimos el archivo comprimido que descargamos en modo "rb" (read binary / lectura binaria).
# Los archivos como imágenes, videos o comprimidos deben leerse en binario, no como texto.
with archivo_comprimido.open("rb") as archivo: 
    # Iniciamos un bucle infinito para leer el archivo por fragmentos.
    while True:  
        # Leemos exactamente 1 Megabyte del archivo.
        bloque = archivo.read(tamano_bloque) 

        # Si 'bloque' está vacío (False), significa que llegamos al final del archivo.
        if not bloque:  
            break  # Salimos del bucle infinito.

        # Alimentamos el objeto verificador con el fragmento leído.
        # Esto va actualizando el cálculo del hash en tiempo real sin desbordar la memoria.
        verificador.update(bloque)  

# Una vez que procesamos todo el archivo, extraemos el resultado final del hash.
# .hexdigest() convierte ese resultado binario en una cadena de texto hexadecimal legible.
md5_calculado = verificador.hexdigest() 

# Mostramos ambos hashes en pantalla para comparación visual.
print("MD5 esperado: ", md5_esperado)
print("MD5 calculado: ", md5_calculado)

# Creamos una variable booleana que será 'True' si ambos hashes son exactamente iguales, o 'False' si difieren.
integridad_correcta = md5_calculado == md5_esperado

# Si la variable es 'False', la integridad falló (el archivo está corrupto o incompleto).
# Lanzamos una excepción para abortar la ejecución, evitando extraer basura.
if not integridad_correcta:
    raise ValueError("La verificación falló. No extraigas el archivo todavía")

# Si el código llega a esta línea sin lanzar errores, la descarga es perfecta.
print("Verificación correcta. El archivo está listo para extraerse.")

# 4. Extraer el conjunto de datos LibriSpeech dev-clean

In [ ]:
# Se importa el módulo nativo tarfile, diseñado para leer y escribir archivos 
# empaquetados (.tar) y comprimidos (como .tar.gz o .tgz).
import tarfile 

# Se realiza una validación de seguridad inicial. Depende de la variable booleana 
# 'integridad_correcta' creada en la celda anterior. Si el archivo descargado 
# estaba corrupto, el programa se aborta aquí antes de intentar extraer basura.
if not integridad_correcta:
    raise ValueError("El archivo debe pasar la verificación antes de extraerse.")

# Definimos la ruta específica donde esperamos encontrar los datos de audio 
# una vez que la extracción finalice correctamente.
carpeta_corpus = carpeta_datos / "LibriSpeech" / "dev-clean"

# Usamos .resolve() para obtener la ruta absoluta y real del directorio de datos.
# Esto es muy importante como medida de seguridad (para evitar el ataque "Zip Slip")
# asegurando que los archivos no se extraigan fuera de esta carpeta base.
destino_base = carpeta_datos.resolve()

# Inicializamos un contador para saber cuántos archivos ya fueron extraídos previamente.
archivos_existentes = 0
# Creamos una lista vacía donde almacenaremos únicamente los archivos que faltan por extraer.
miembros_pendientes = []

# Abrimos el archivo comprimido usando el bloque 'with' (para que se cierre automáticamente).
# mode="r:gz" indica que vamos a Leer (r) un archivo comprimido con Gzip (gz).
with tarfile.open(archivo_comprimido, mode= "r:gz") as archivo:
    # Obtenemos la lista completa de todo lo que contiene el archivo comprimido 
    # (carpetas, subcarpetas, archivos de texto, audios, etc.).
    miembros = archivo.getmembers()

    # Iniciamos un bucle para revisar, uno por uno, cada elemento dentro del archivo.
    for miembro in miembros:
        # Construimos la ruta donde este elemento específico debería guardarse en nuestro disco duro
        # y calculamos su ruta absoluta con .resolve().
        ruta_destino = (carpeta_datos / miembro.name).resolve()

        # VALIDACIÓN DE SEGURIDAD (Prevención de Path Traversal):
        # Verificamos que la ruta de destino planeada esté estrictamente contenida dentro 
        # de nuestra carpeta 'destino_base'. Esto bloquea archivos maliciosos que tengan 
        # nombres como "../../../etc/password".
        if destino_base not in ruta_destino.parents:
            raise ValueError(f"Ruta de extracción no permitida:  {miembro.name}")

        # Si el elemento que estamos revisando es un directorio (carpeta):
        if miembro.isdir():
            # Creamos la carpeta físicamente en el disco duro.
            ruta_destino.mkdir(parents = True, exist_ok = True)
            continue # Saltamos a la siguiente iteración del bucle, ya que las carpetas no se extraen como archivos.

        # Si el elemento no es una carpeta, validamos que sí sea un archivo normal (isfile).
        # Esto filtra enlaces simbólicos u otros elementos raros que no nos interesan.
        if not miembro.isfile():
            raise ValueError(f"Tipo de archivo no permitido: {miembro.name}")

        # Comprobamos si el archivo ya existe físicamente en nuestro disco duro (de una extracción pasada).
        if ruta_destino.exists():
            # Si existe, consultamos su tamaño actual en bytes en nuestro disco duro.
            tamano_actual = ruta_destino.stat().st_size

            # Lo comparamos con el tamaño que el archivo declara tener dentro del .tar.gz (miembro.size).
            # Si los tamaños no coinciden, el archivo existente está incompleto o corrupto.
            if tamano_actual != miembro.size:
                raise ValueError(f"El tamaño del archivo no coincide: {ruta_destino}")

            # Si el tamaño coincide, asumimos que el archivo ya está extraído correctamente y aumentamos el contador.
            archivos_existentes += 1
        
        else:
            # Si el archivo NO existe en nuestro disco duro, lo añadimos a la lista de pendientes por extraer.
            miembros_pendientes.append(miembro)

    # Una vez revisados todos los elementos, comprobamos si la lista de pendientes tiene algo.
    if miembros_pendientes:
        print("Extratendo los archivos faltantes...")

        # Ordenamos la extracción masiva, pero pasándole ÚNICAMENTE la lista de los archivos 
        # que nos faltan (miembros_pendientes). Esto ahorra muchísimo tiempo si el proceso 
        # se había interrumpido antes y lo volvemos a ejecutar.
        archivo.extractall(
            path= carpeta_datos,
            members = miembros_pendientes)

# Finalizada la extracción, verificamos que la estructura principal de carpetas del dataset 
# (LibriSpeech/dev-clean) realmente exista y sea un directorio válido.
if not carpeta_corpus.is_dir():
    raise FileNotFoundError("No se encontró la carpeta del corpus.")

# Imprimimos un resumen en consola de lo que el algoritmo hizo.
print("Archivos existentes con tamaño correcto:", archivos_existentes)
print("Archivos extraídos: ", len(miembros_pendientes))
print("Comprobación de extracción finalizada.")

# 5. Contar los locutores y las grabaciones

In [ ]:
# Verificamos nuevamente que la carpeta del corpus exista y sea un directorio válido.
# Si no existe, lanzamos un error y detenemos la ejecución.
if not carpeta_corpus.is_dir():
    raise FileNotFoundError("No se encontó la carpeta del corpus")

# Creamos una lista vacía para almacenar las rutas de las carpetas correspondientes a cada locutor.
carpetas_locutores = []

# Iteramos sobre todos los elementos dentro de la carpeta del corpus usando .iterdir()
for carpeta in carpeta_corpus.iterdir():
    # Filtramos: solo nos interesan los elementos que sean directorios (carpetas) 
    # Y cuyo nombre esté compuesto únicamente por números (.isdigit()). 
    # En LibriSpeech, cada locutor tiene una carpeta nombrada con su ID numérico.
    if carpeta.is_dir() and carpeta.name.isdigit():
        carpetas_locutores.append(carpeta)

# Definimos una función auxiliar que toma la ruta de una carpeta y devuelve su nombre 
# convertido a un número entero. Esto nos servirá para ordenar las carpetas numéricamente.
def obtener_identificador(carpeta):
    return int(carpeta.name)

# Ordenamos la lista de carpetas de locutores utilizando la función anterior como criterio (key).
# Así, la carpeta "2" irá antes que la "10", en lugar de ordenarse alfabéticamente.
carpetas_locutores.sort(key = obtener_identificador)

# Inicializamos una lista para guardar un diccionario con la información de cada locutor.
resumen_locutores = []
# Inicializamos un contador global para saber cuántas grabaciones hay en total en todo el dataset.
total_grabaciones = 0

# Iniciamos un bucle para procesar cada carpeta de locutor (ya ordenadas).
for carpeta_locutor in carpetas_locutores:
    # Usamos .rglob("*.flac") (recursive glob) para buscar todos los archivos que terminen en .flac 
    # dentro de la carpeta del locutor y en todas sus posibles subcarpetas.
    # Convertimos el resultado a una lista para poder contar sus elementos.
    archivos_locutor = list(carpeta_locutor.rglob("*.flac"))
    
    # Obtenemos la cantidad de grabaciones (archivos .flac) que tiene este locutor específico.
    cantidad_grabaciones = len(archivos_locutor)

    # Creamos un diccionario llamado 'registro' para organizar la información de este locutor.
    registro ={
        "locutor": carpeta_locutor.name, # El ID del locutor (nombre de la carpeta)
        "grabaciones": cantidad_grabaciones, # Cuántos audios tiene
    }

    # Agregamos este diccionario a nuestra lista de resumen general.
    resumen_locutores.append(registro)
    # Sumamos las grabaciones de este locutor al contador global.
    total_grabaciones += cantidad_grabaciones

# Obtenemos la cantidad total de locutores basándonos en cuántas carpetas encontramos.
numero_locutores = len(carpetas_locutores)

# Comenzamos a imprimir un reporte en la consola.
print("Número de locutores:", numero_locutores)
# Utilizamos f-strings con formato de alineación para crear el encabezado de una tabla.
# '<12' alinea a la izquierda ocupando 12 espacios, y '>12' alinea a la derecha ocupando 12 espacios.
print(f"{'Locutor':<12}{'Grabaciones':>12}")
# Imprimimos una línea de puntos para separar el encabezado de los datos.
print("."*24)

# Recorremos la lista de diccionarios que creamos anteriormente para imprimir cada fila.
for registro in resumen_locutores:
    identificador = registro["locutor"]
    cantidad = registro["grabaciones"]

    # Imprimimos los datos de cada locutor aplicando el mismo formato de columnas que en el encabezado.
    print(f"{identificador:<12}{cantidad:>12}")

# Validaciones finales de consistencia de los datos:
# El dataset dev-clean de LibriSpeech debe tener exactamente 40 locutores.
# Si encontramos un número diferente, levantamos un error indicando que los datos podrían estar incompletos.
if numero_locutores != 40:
    raise ValueError(f"Se esperaban 40 locutores, pero no se encontraron {numero_locutores}.")

# Si el total de grabaciones es cero, significa que no se extrajeron los audios correctamente.
if total_grabaciones == 0:
    raise ValueError("No se encotraron grabaciones FLAC.")

# 6. Crear el inventario de metadatos de los audios 

In [ ]:
# Importamos pandas, la biblioteca estándar de Python para análisis y manipulación de datos.
# La usamos para crear y manejar estructuras tabulares llamadas DataFrames.
import pandas as pd
# Importamos soundfile, una biblioteca especializada en leer y escribir archivos de audio.
# Nos permitirá extraer la información técnica de los audios sin tener que cargarlos completamente en memoria.
import soundfile as sf

# Importamos la función 'display' específica para entornos como Jupyter Notebooks o Google Colab.
# Permite renderizar y mostrar DataFrames (tablas) de forma mucho más limpia y visual que un simple 'print'.
from IPython.display import display

# Buscamos todos los archivos .flac dentro de la carpeta del corpus de forma recursiva (.rglob).
# Usamos sorted() para ordenar alfabéticamente la lista de rutas obtenidas y mantener la consistencia.
archivos_audio = sorted(carpeta_corpus.rglob("*.flac"))

# Inicializamos dos listas vacías. 
# Una para guardar los metadatos extraídos con éxito y otra para registrar los archivos que fallen.
registros_audio = []
errores_lectura = []

# Iniciamos un bucle para recorrer cada ruta de archivo de audio encontrada.
for ruta_audio in archivos_audio:
    # Usamos un bloque try-except porque leer archivos físicos siempre es propenso a errores 
    # (archivos corruptos, permisos denegados, etc.).
    try:
        # sf.info() lee exclusivamente las cabeceras (metadatos) del archivo de audio, lo cual es muy rápido.
        # Convertimos 'ruta_audio' (objeto Path) a string puro porque soundfile lo requiere así.
        informacion = sf.info(str(ruta_audio))

        # Creamos un diccionario recolectando toda la información que nos interesa del archivo y su estructura.
        registro = {
            "archivo": ruta_audio.name, # El nombre del archivo con su extensión (ej. 1234-5678-0001.flac)
            # La jerarquía de LibriSpeech es: locutor/capítulo/archivo.flac
            # Por lo tanto, el "abuelo" de la ruta del archivo es el ID del locutor.
            "locutor": ruta_audio.parent.parent.name, 
            # El "padre" de la ruta del archivo es el ID del capítulo del audiolibro.
            "capítulo": ruta_audio.parent.name,
            "ruta": str(ruta_audio), # La ruta absoluta en el disco para poder cargar el audio más adelante
            "duracion_segundos": informacion.duration, # Duración en segundos
            "frecuencia_muestreo": informacion.samplerate, # Tasa de muestreo (ej. 16000 Hz)
            "canales": informacion.channels, # Número de canales (1 para Mono, 2 para Estéreo)
            "numero_muestras":informacion.frames, # Cantidad total de puntos de datos en el audio
            "formato": informacion.format, # Formato contenedor (ej. FLAC)
            "subtipo": informacion.subtype # Subtipo de codificación de los datos
        }

        # Añadimos nuestro diccionario con los datos a la lista de registros exitosos.
        registros_audio.append(registro)
        
    # Si ocurre un error del sistema operativo (OSError) o de ejecución (RuntimeError) al leer el archivo:
    except (OSError, RuntimeError) as error:
        # Creamos un diccionario almacenando la ruta que falló y el mensaje del error exacto.
        registro_error = {
            "archivo": str(ruta_audio),
            "error": str(error)
        }

        # Añadimos el registro fallido a nuestra lista de errores.
        errores_lectura.append(registro_error)

# Convertimos la lista de diccionarios exitosos en un DataFrame (una tabla estructurada) usando pandas.
inventario_audio = pd.DataFrame(registros_audio)
# Hacemos lo mismo con la lista de errores, especificando explícitamente el nombre de las columnas.
tabla_errores = pd.DataFrame(
    errores_lectura,
    columns=["archivo","error"])

# Imprimimos un resumen cuantitativo en la consola.
print("Archivos encontrados:",len(archivos_audio))
print("Metadatos leídos:", len(inventario_audio))
print("Errores de lectura:", len(tabla_errores))

# Mostramos en pantalla las primeras 5 filas (.head()) de nuestra tabla principal de audios
# usando la función 'display' para que se vea con un formato agradable de tabla en el notebook.
display(inventario_audio.head())

# Si nuestra tabla de errores NO está vacía, la mostramos en pantalla para analizar qué falló.
if not tabla_errores.empty:
    display(tabla_errores)

# Validación final crítica: si la tabla principal está vacía, significa que falló la lectura 
# de absolutamente todos los archivos. Lanzamos un error deteniendo la ejecución.
if inventario_audio.empty:
    raise ValueError("No se pudieron leer los metadatos de ninguna grabación.")

# 7. Resumir la duración y los formatos del corpus

In [ ]:
# Obtenemos la cantidad total de grabaciones midiendo la longitud (número de filas) 
# de nuestro DataFrame 'inventario_audio'.
cantidad_grabaciones = len(inventario_audio)

# Calculamos el número de locutores únicos.
# Seleccionamos la columna "locutor" del DataFrame y usamos .nunique() (Number of Unique),
# que cuenta cuántos valores diferentes existen en esa columna, sin contar duplicados.
cantidad_locutores = inventario_audio["locutor"].nunique()

# Calculamos la duración total del dataset en segundos.
# Seleccionamos la columna "duracion_segundos" y aplicamos .sum() para sumar todos sus valores.
duracion_total_segundos = inventario_audio["duracion_segundos"].sum()

# Convertimos la duración total de segundos a minutos dividiendo entre 60.
duracion_total_minutos = duracion_total_segundos / 60
# Convertimos la duración de minutos a horas dividiendo nuevamente entre 60.
duracion_total_horas = duracion_total_minutos/60

# Imprimimos en consola el resumen general del corpus.
print("Locutores:", cantidad_locutores)
print("Grabaciones:", cantidad_grabaciones)
# Usamos f-strings para formatear la salida mostrando solo dos decimales (.2f).
print(f"Duración total en minutos: {duracion_total_minutos:.2f}")
print(f"Duración total en horas: {duracion_total_horas:.2f}")

# El método .describe() de pandas genera automáticamente estadísticas descriptivas 
# de una columna numérica (conteo, media, desviación estándar, mínimo, máximo y cuartiles).
estadisticas_duracion = inventario_audio["duracion_segundos"].describe()

print("\nEstadísticas de duración por grabación, en segundos:")
# Como .describe() devuelve un objeto de tipo Series, usamos .to_frame() para convertirlo 
# en un DataFrame de una sola columna (nombrada "duracion_segundos").
# Luego, usamos display() para mostrarlo con el formato de tabla estilizada del notebook.
display(estadisticas_duracion.to_frame(name="duracion_segundos"))

# Definimos una lista con los nombres de las columnas que contienen la información técnica del formato.
columnas_formato = [
    "frecuencia_muestreo",
    "canales",
    "formato",
    "subtipo"
]

# Agrupamos los datos del inventario basándonos en estas 4 columnas técnicas.
# .groupby() junta todas las filas que tengan exactamente la misma combinación de estos 4 valores.
grupos_formato = inventario_audio.groupby(columnas_formato)

# Usamos .size() sobre los grupos creados para contar cuántas grabaciones (filas) 
# existen dentro de cada grupo (es decir, cuántos audios comparten el mismo formato).
# Esto devuelve una Serie de pandas con un índice múltiple.
resumen_formatos = grupos_formato.size()

# .reset_index() convierte el índice múltiple nuevamente en columnas normales de un DataFrame.
# El parámetro name="grabaciones" asigna ese nombre a la nueva columna que contiene los conteos 
# calculados por .size().
resumen_formatos = resumen_formatos.reset_index(name = "grabaciones")

print("\nFormatos presentes en el corpus:")
# Mostramos la tabla resumen final donde se ven las combinaciones de formatos y cuántos audios hay de cada una.
display(resumen_formatos)